# V2: pretrained BERT disfluency restoration

Fine-tune **bert-base-uncased** on the fixed v2 Switchboard split. **Disfluency F1** selects the checkpoint and decoder and is the reported metric.

Select **Runtime → Change runtime type → GPU** (L4 if available, otherwise T4). Run all cells. When prompted, upload `switchboard_v2_colab.zip` alongside this notebook. Three epochs over all training windows; no compact-model weights are reused. Checkpoints/logs stay in the Colab VM until downloaded; closing the page does not guarantee the VM will remain allocated.

Inputs: flattened transcript with dictionary pronunciation guidance + full spoken phone sequence. Phones are dictionary-derived with simulated errors, not measured recognizer output.

In [ ]:
# 1. Verify GPU
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU, then Run all again.'
print('GPU:', torch.cuda.get_device_name(0))
print('Memory GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))

In [ ]:
# 2. Upload the fixed dataset and source bundle
from google.colab import files
from pathlib import Path
import hashlib, zipfile
archive = Path('/content/switchboard_v2_colab.zip')
if not archive.exists():
    uploaded = files.upload()
    assert 'switchboard_v2_colab.zip' in uploaded, 'Upload switchboard_v2_colab.zip'
assert hashlib.sha256(archive.read_bytes()).hexdigest() == 'bc3cbdb61dfe21c3ddf6546cf3c38668c174d51de417be94763f255f354463fc', 'Bundle checksum mismatch'
PROJECT = Path('/content/cl2_v2')
PROJECT.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    z.extractall(PROJECT)
print('Verified bundle and unpacked fixed training/dev/test data.')

In [ ]:
# 3. Install model dependencies
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'transformers==5.22.2', 'jiwer', 'pyyaml'])
print('Dependencies ready.')

In [ ]:
# 4. Train in a detached process and watch its log
# Re-run this cell after a page reconnect to watch the same process.
import os, subprocess, sys, time, json
PROJECT = Path('/content/cl2_v2')
OUT = PROJECT / 'v2/checkpoints/pretrained_colab'
OUT.mkdir(parents=True, exist_ok=True)
LOG = OUT / 'training.log'
PID = OUT / 'training.pid'
DONE = OUT / 'disfluency_f1_results.json'

def alive():
    if not PID.exists(): return False
    try:
        command = Path('/proc/' + PID.read_text().strip() + '/cmdline').read_bytes()
        return b'v2/train_colab.py' in command
    except (FileNotFoundError, ProcessLookupError):
        return False

if not DONE.exists() and not alive():
    cmd = [sys.executable, '-u', 'v2/train_colab.py', '--epochs', '3']
    if (OUT / 'latest.pt').exists(): cmd.append('--resume')
    proc = subprocess.Popen(cmd, cwd=PROJECT, env=dict(os.environ, PYTHONUNBUFFERED='1', TOKENIZERS_PARALLELISM='false'),
                            stdout=open(LOG, 'a'), stderr=subprocess.STDOUT, start_new_session=True)
    PID.write_text(str(proc.pid))
    print('Started GPU training, PID', proc.pid, flush=True)
else:
    print('Existing run:', 'completed' if DONE.exists() and not alive() else 'running', flush=True)
last = ''
while alive():
    lines = LOG.read_text(errors='replace').splitlines() if LOG.exists() else []
    current = '\n'.join(lines[-3:])
    if current != last:
        print(current, flush=True); last = current
    time.sleep(15)
print('\n'.join(LOG.read_text(errors='replace').splitlines()[-20:]))
assert DONE.exists() and (OUT / 'disfluency_f1_report.md').exists(), 'Run did not finish successfully; inspect training.log.'

In [ ]:
# 5. Display disfluency F1 results
import json
from IPython.display import display, Markdown
results = json.loads((OUT / 'disfluency_f1_results.json').read_text())
display(Markdown((OUT / 'disfluency_f1_report.md').read_text()))
print('Best epoch:', results['best_epoch'], '| STOP offset:', results['stop_offset'])

In [ ]:
# 6. Download weights, predictions, metrics, and the report
import zipfile
from google.colab import files
result_zip = Path('/content/v2_pretrained_bert_results.zip')
with zipfile.ZipFile(result_zip, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.rglob('*')):
        if p.is_file() and p.name not in {'latest.pt', 'training.pid'}:
            z.write(p, Path('pretrained_colab') / p.relative_to(OUT))
print('Result archive MiB:', round(result_zip.stat().st_size/2**20, 1))
files.download(str(result_zip))